In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from joblib import dump

# Add project root
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features

In [2]:

df_feats, feature_cols = get_features("../data/raw")

# Filter seasons and minutes
df_ml = df_feats[
    (df_feats["season_end_year"] >= 2008) &
    (df_feats["season_end_year"] <= 2024) &
    (df_feats["minutes_played"] >= 100)
].copy()

print("Full ML dataset:", df_ml.shape)


Full ML dataset: (23442, 79)


In [3]:
final_features = [
    # === Selected 30 ===
    'a_per90_z_lag1', 'ga_per90_z_lag1', 'matches_played_z_lag1',
    'g_per90_z_lag1', 'pen_share_z_lag1', 'g_per90_w', 'ga_per90_w',
    'a_per90_w', 'pen_share_w', 'a_per90_z_delta', 'ga_per90_z_delta',
    'main_position', 'g_per90_z_delta', 'age', 'win_rate', 'height',
    'goals_per_game', 'minutes_played_z_lag1', 'pen_share_z_delta',
    'age_norm', 'team_ucl_strength', 'age_penalty',
    'matches_played_z_delta', 'gc_per90_z_lag1',
    'minutes_played_z_delta', 'season_end_year',
    'Titles', 'num_trophies', 'won_champions',

    # === Metadata (not scaled) ===
    'player_id', 'player_name'
]

# No duplicados
final_features = list(dict.fromkeys(final_features))

# Metadatos que NO entran al modelo, sólo para reporting / outputs
metadata_cols = [
    "player_id",
    "player_name",
    "season_end_year",
    "minutes_played",   # la añadimos aunque no esté en final_features
]

# Aseguramos que existen en df_ml
missing = [c for c in final_features if c not in df_ml.columns]
if missing:
    print("⚠ WARNING: these final_features are missing in df_ml:", missing)

available_final = [c for c in final_features if c in df_ml.columns]

# Features que sí usará el modelo (sin metadatos)
model_feature_cols = [
    c for c in available_final
    if c not in metadata_cols
]

print("Num model features:", len(model_feature_cols))
print("Model features:", model_feature_cols)


Num model features: 28
Model features: ['a_per90_z_lag1', 'ga_per90_z_lag1', 'matches_played_z_lag1', 'g_per90_z_lag1', 'pen_share_z_lag1', 'g_per90_w', 'ga_per90_w', 'a_per90_w', 'pen_share_w', 'a_per90_z_delta', 'ga_per90_z_delta', 'main_position', 'g_per90_z_delta', 'age', 'win_rate', 'height', 'goals_per_game', 'minutes_played_z_lag1', 'pen_share_z_delta', 'age_norm', 'team_ucl_strength', 'age_penalty', 'matches_played_z_delta', 'gc_per90_z_lag1', 'minutes_played_z_delta', 'Titles', 'num_trophies', 'won_champions']


In [6]:
df_train = df_ml[df_ml["season_end_year"] <= 2018].copy()
df_val   = df_ml[(df_ml["season_end_year"] >= 2019) &
                 (df_ml["season_end_year"] <= 2022)].copy()

print("Train:", df_train.shape)
print("Val:", df_val.shape)

X_train = df_train[model_feature_cols].copy()
y_train = df_train["ballon_dor_winner"].astype(int)

X_val = df_val[model_feature_cols].copy()
y_val = df_val["ballon_dor_winner"].astype(int)

print("Train class balance:\n", y_train.value_counts())


Train: (14301, 79)
Val: (6038, 79)
Train class balance:
 ballon_dor_winner
0    14290
1       11
Name: count, dtype: int64


In [7]:

numeric_features = [
    c for c in model_feature_cols
    if df_ml[c].dtype != "object"
]

categorical_features = [
    c for c in model_feature_cols
    if df_ml[c].dtype == "object"
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop",
)

Numeric features: ['a_per90_z_lag1', 'ga_per90_z_lag1', 'matches_played_z_lag1', 'g_per90_z_lag1', 'pen_share_z_lag1', 'g_per90_w', 'ga_per90_w', 'a_per90_w', 'pen_share_w', 'a_per90_z_delta', 'ga_per90_z_delta', 'g_per90_z_delta', 'age', 'win_rate', 'height', 'goals_per_game', 'minutes_played_z_lag1', 'pen_share_z_delta', 'age_norm', 'team_ucl_strength', 'age_penalty', 'matches_played_z_delta', 'gc_per90_z_lag1', 'minutes_played_z_delta', 'Titles', 'num_trophies', 'won_champions']
Categorical features: ['main_position']


In [8]:

models = {
    "lr": LogisticRegression(
        max_iter=2000,
        n_jobs=-1
    ),
    "rf": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ),
    "xgb": XGBClassifier(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
}

pipelines = {}
val_results = {}

for name, clf in models.items():
    print(f"\n===== TRAINING {name.upper()} PIPELINE =====")

    pipe = ImbPipeline(steps=[
        ("preprocess", preprocessor),
        ("smote", SMOTE(k_neighbors=1, random_state=42)),
        ("clf", clf),
    ])

    # MUY IMPORTANTE:
    # fit() aprende:
    # - scaler + OHE SOLO con X_train
    # - SMOTE SOLO con X_train, y_train
    # No toca X_val → NO HAY DATA LEAKAGE
    pipe.fit(X_train, y_train)

    proba_val = pipe.predict_proba(X_val)[:, 1]
    pred_val_default = (proba_val >= 0.5).astype(int)

    auc_val = roc_auc_score(y_val, proba_val)
    rec_val = recall_score(y_val, pred_val_default, zero_division=0)
    prec_val = precision_score(y_val, pred_val_default, zero_division=0)
    f1_val = f1_score(y_val, pred_val_default, zero_division=0)

    val_results[name] = {
        "AUC": auc_val,
        "Recall": rec_val,
        "Precision": prec_val,
        "F1": f1_val,
    }

    pipelines[name] = pipe

    print(f"AUC       : {auc_val:.6f}")
    print(f"Recall    : {rec_val:.4f}")
    print(f"Precision : {prec_val:.4f}")
    print(f"F1        : {f1_val:.4f}")

print("\nValidation summary:", val_results)


===== TRAINING LR PIPELINE =====
AUC       : 0.999779
Recall    : 1.0000
Precision : 0.3000
F1        : 0.4615

===== TRAINING RF PIPELINE =====
AUC       : 0.999061
Recall    : 0.0000
Precision : 0.0000
F1        : 0.0000

===== TRAINING XGB PIPELINE =====
AUC       : 0.999669
Recall    : 0.3333
Precision : 0.3333
F1        : 0.3333

Validation summary: {'lr': {'AUC': 0.9997790665562, 'Recall': 1.0, 'Precision': 0.3, 'F1': 0.46153846153846156}, 'rf': {'AUC': 0.9990610328638498, 'Recall': 0.0, 'Precision': 0.0, 'F1': 0.0}, 'xgb': {'AUC': 0.9996685998342999, 'Recall': 0.3333333333333333, 'Precision': 0.3333333333333333, 'F1': 0.3333333333333333}}


In [9]:

best_model_name = max(val_results, key=lambda k: val_results[k]["AUC"])
best_pipe = pipelines[best_model_name]

print("\n=======================================")
print("BEST MODEL ON VALIDATION:", best_model_name.upper())
print(val_results[best_model_name])
print("=======================================\n")


BEST MODEL ON VALIDATION: LR
{'AUC': 0.9997790665562, 'Recall': 1.0, 'Precision': 0.3, 'F1': 0.46153846153846156}



In [10]:

from sklearn.metrics import precision_recall_curve

proba_val_best = best_pipe.predict_proba(X_val)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_val, proba_val_best)

# ejemplo: escoger threshold que maximiza F1 en validación
best_f1 = -1
best_thr = 0.5

for thr in thresholds:
    preds_thr = (proba_val_best >= thr).astype(int)
    f1_thr = f1_score(y_val, preds_thr, zero_division=0)
    if f1_thr > best_f1:
        best_f1 = f1_thr
        best_thr = thr

print(f"Best threshold by F1 on validation: {best_thr:.4f} (F1 = {best_f1:.4f})")


Best threshold by F1 on validation: 0.9873 (F1 = 0.6667)
